# BrownDye2: PQR/APBS preparation

Prepare a docked protein-ligand pose for BrownDye2. BrownDye needs one PQR/XML file and one APBS electrostatic map for each independently moving rigid body, so the workflow writes separate protein and ligand inputs rather than a single bound-complex map.

**This notebook**: prepare protein and ligand inputs from a docked PDB.
1. Fix protein with PDBFixer (add missing atoms, protonate at target pH)
2. Assign ligand bond orders from SMILES, write SDF for antechamber
3. Write ligand metadata consumed by the shell scripts

**Shell scripts**:
1. `run_ambertools.sh` -- assign Amber/GAFF2 charges and radii, write protein/ligand PQRs
2. `run_apbs.sh` -- solve APBS separately for `protein.pqr` and `ligand.pqr`
3. `bdprep.sh` -- convert PQR to BrownDye XML, define reaction criteria, and run `bd_top`
4. `bdrun.sh` -- run BrownDye and estimate association rates

In [ ]:
from pathlib import Path

from Bio.PDB import PDBIO, PDBParser
from rdkit import Chem

from mdpp.prep import ChainSelect, assign_topology, fix_pdb, run_propka

COMPLEX_PDB = Path("TurboID-bioAMP_model_0.pdb")
WORKDIR = Path("tmp")
WORKDIR.mkdir(exist_ok=True)

# Canonical SMILES of the ligand (used to assign bond orders)
LIGAND_SMILES = r"Nc1ncnc2n(cnc12)[C@@H]3O[C@H](CO[P]([O-])(=O)OC(=O)CCCC[C@@H]4SC[C@@H]5NC(=O)N[C@H]45)[C@@H](O)[C@H]3O"
PROTEIN_CHAIN_ID = "A"
LIGAND_CHAIN_ID = "B"
PH = 7.4

## Step 1: Verify protonation and fix protein

Extract the protein chain, run PropKa as an explicit pKa/protonation-state check at the target pH, then add missing residues/atoms/hydrogens via PDBFixer.

In [ ]:
parser = PDBParser(QUIET=True)
structure = parser.get_structure("complex", str(COMPLEX_PDB))
model = structure[0]
chains = {chain.id: chain for chain in model}

pdb_io = PDBIO()
pdb_io.set_structure(structure)

# Extract protein chain
protein_pdb = WORKDIR / "protein.pdb"
pdb_io.save(str(protein_pdb), ChainSelect(PROTEIN_CHAIN_ID))

# Verify protonation with PropKa before PDBFixer assigns hydrogens.
propka_result = run_propka(protein_pdb)
nonstandard = propka_result.get_nonstandard(PH)
propka_report = WORKDIR / "protein_propka.tsv"
with propka_report.open("w") as f:
    f.write(
        "residue_type\tres_num\tchain_id\tpka\tmodel_pka\tpropka_protonated\tmodel_protonated\n"
    )
    for residue in propka_result.residues:
        f.write(
            f"{residue.residue_type}\t{residue.res_num}\t{residue.chain_id}\t"
            f"{residue.pka:.3f}\t{residue.model_pka:.3f}\t"
            f"{residue.is_protonated_at(PH)}\t{residue.is_default_protonated_at(PH)}\n"
        )

print(f"PropKa residues checked: {len(propka_result.residues)} -> {propka_report}")
if nonstandard:
    print(f"PropKa predicts {len(nonstandard)} non-standard protonation state(s) at pH {PH}:")
    for residue in nonstandard:
        print(
            f"  {residue.label}: pKa={residue.pka:.2f}, model={residue.model_pka:.2f}, "
            f"PropKa={'protonated' if residue.is_protonated_at(PH) else 'deprotonated'}, "
            f"model={'protonated' if residue.is_default_protonated_at(PH) else 'deprotonated'}"
        )
else:
    print(f"PropKa agrees with model-pKa defaults for all titratable residues at pH {PH}.")

protein_fixed_pdb = WORKDIR / "protein_fixed.pdb"
fix_pdb(protein_pdb, protein_fixed_pdb, pH=PH)
print(f"Fixed protein -> {protein_fixed_pdb}")

## Step 2: Assign ligand topology

Extract ligand chain, assign bond orders from a SMILES template, and write an SDF
for antechamber. The SMILES is needed because PDB files lack bond-order information.

In [ ]:
# Extract ligand chain to PDB for RDKit parsing
ligand_pdb = WORKDIR / "ligand.pdb"
pdb_io.save(str(ligand_pdb), ChainSelect(LIGAND_CHAIN_ID))

# Assign bond orders from SMILES template
template_mol = Chem.MolFromSmiles(LIGAND_SMILES, sanitize=True)
ligand_net_charge = Chem.GetFormalCharge(template_mol)
print(f"Ligand net charge: {ligand_net_charge}")

mol = Chem.MolFromPDBFile(str(ligand_pdb), sanitize=True, removeHs=True)
mol_assigned = assign_topology(mol, template_mol)

# Write SDF for antechamber and metadata for the shell scripts
lig_resnames = sorted({res.get_resname().strip() for res in chains[LIGAND_CHAIN_ID].get_residues()})
lig_resname = lig_resnames[0]
mol_assigned.SetProp("_Name", lig_resname)

ligand_sdf = WORKDIR / "ligand.sdf"
with Chem.SDWriter(str(ligand_sdf)) as w:
    w.write(mol_assigned)

(WORKDIR / "ligand_charge.txt").write_text(f"{ligand_net_charge}\n")
(WORKDIR / "ligand_resname.txt").write_text(f"{lig_resname}\n")
print(f"Ligand SDF -> {ligand_sdf}")
print(f"Ligand metadata -> {WORKDIR / 'ligand_charge.txt'}, {WORKDIR / 'ligand_resname.txt'}")

## Next: run the shell workflow

The notebook produced `tmp/protein_fixed.pdb`, `tmp/ligand.sdf`, and ligand metadata. The shell scripts pick up from here: they load the protein and ligand separately into Amber/tleap, write separate PQR files, solve APBS separately for each BrownDye rigid body, and then generate BrownDye XML inputs.

```bash
conda activate ambertools
cd examples/browndye
bash run_ambertools.sh
bash run_apbs.sh
bash bdprep.sh
bash bdrun.sh
```

Key outputs in `tmp/`:

| File | Description |
|------|-------------|
| `protein.pqr`, `ligand.pqr` | BrownDye rigid-body PQR inputs |
| `protein.dx`, `ligand.dx` | APBS electrostatic maps for each rigid body |
| `protein_atoms.xml`, `ligand_atoms.xml` | BrownDye XML atom files |
| `reaction_pairs.xml`, `reactions.xml` | Contact-based reaction criteria from the docked pose |
| `protein_ligand_simulation.xml` | BrownDye simulation input generated by `bd_top` |
| `results.xml`, `rate_constant.txt` | BrownDye simulation results and rate estimate |